<a href="https://colab.research.google.com/github/kyahikaru/hinglish-prompt-injection-detector/blob/main/notebooks/srinivasan_dataset_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers scikit-learn pandas numpy openpyxl -q

In [3]:
from google.colab import files
print("Upload these two files:")
print("1. PromptInjectionPrompts.xlsx (Srinivasan dataset)")
print("2. adversarial_test_set_v2.json (your 100 stealth samples)")
uploaded = files.upload()

Upload these two files:
1. PromptInjectionPrompts.xlsx (Srinivasan dataset)
2. adversarial_test_set_v2.json (your 100 stealth samples)


Saving adversarial_test_set_v2.json to adversarial_test_set_v2.json
Saving PromptInjectionPrompts.xlsx to PromptInjectionPrompts (1).xlsx


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel('PromptInjectionPrompts.xlsx')
df = df.rename(columns={'Prompts': 'text', 'Label': 'label'})

train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)

print("--- SRINIVASAN DATASET SPLIT ---")
print(f"Train samples: {len(train_df)}")
print(f"Test samples : {len(test_df)}")

--- SRINIVASAN DATASET SPLIT ---
Train samples: 3200
Test samples : 800


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

X_train = model.encode(train_df['text'].tolist(), show_progress_bar=True)
X_test  = model.encode(test_df['text'].tolist(), show_progress_bar=True)

y_train = train_df['label'].values
y_test  = test_df['label'].values

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)

print("--- BASELINE METRICS ON SRINIVASAN TEST SET ---")
print(classification_report(y_test, clf.predict(X_test)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

--- BASELINE METRICS ON SRINIVASAN TEST SET ---
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       400
           1       0.94      0.94      0.94       400

    accuracy                           0.94       800
   macro avg       0.94      0.94      0.94       800
weighted avg       0.94      0.94      0.94       800



In [8]:
import json

with open('adversarial_test_set_v2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

samples = data['samples'] if 'samples' in data else data
print(f"Loaded {len(samples)} stealth adversarial prompts")

Loaded 100 stealth adversarial prompts


In [9]:
print("=== SRINIVASAN BASELINE vs 100 STEALTH SAMPLES ===")

blocked = 0
results = []

for sample in samples:
    text = sample['hinglish']
    emb = model.encode([text], convert_to_numpy=True).astype('float32')
    pred = clf.predict(emb)[0]
    status = "BLOCKED" if pred == 1 else "BYPASSED"
    if pred == 1:
        blocked += 1
    results.append({
        "id": sample["id"],
        "category": sample["category"],
        "prompt": text[:80] + ("..." if len(text) > 80 else ""),
        "srinivasan": status
    })

summary_df = pd.DataFrame(results)
print(summary_df.groupby("category").size().reset_index(name="count"))

print(f"\nFINAL RESULT:")
print(f"Srinivasan-style model blocked : {blocked}/100 ({blocked:.1f}%)")
print(f"Bypassed (failed)              : {100 - blocked}/100")

=== SRINIVASAN BASELINE vs 100 STEALTH SAMPLES ===
                   category  count
0          academic_masking      7
1   authority_impersonation      7
2          direct_jailbreak      6
3           fiction_framing     11
4        gradual_escalation      9
5              mixed_script      9
6                obfuscated      6
7              pure_english     17
8                pure_hindi     17
9          research_framing      7
10         stealth_academic      1
11          stealth_fiction      1
12         stealth_research      2

FINAL RESULT:
Srinivasan-style model blocked : 80/100 (80.0%)
Bypassed (failed)              : 20/100
